# 03b - GPU-intensive Experiments (C/D)

## Overview
This notebook runs GPU-intensive experiments using **MLP** and **FT-Transformer** for carbon emission prediction with transfer learning.

### Experiment Types
| Code | Experiment Name | Description | Model |
|------|----------------|-------------|-------|
| **C** | Transfer Learning | Global pre-training → single-country fine-tuning | MLP |
| **C_dev** | Developed Countries Transfer Learning | Developed countries pre-training → fine-tuning | MLP |
| **D_lora** | FT-Transformer + LoRA | Global pre-training → LoRA fine-tuning | FT-Transformer |
| **D_lora_dev** | FT-Transformer + LoRA (Developed) | Developed countries pre-training → LoRA fine-tuning | FT-Transformer |
| **D_full** | FT-Transformer Full Fine-tuning | Global pre-training → full parameter fine-tuning | FT-Transformer |
| **D_full_dev** | FT-Transformer Full (Developed) | Developed countries pre-training → full fine-tuning | FT-Transformer |

### Hardware Requirements
- **GPU**: NVIDIA GPU with CUDA support recommended
- **Memory**: 16GB+ GPU memory recommended for FT-Transformer
- **CPU**: Multi-core as fallback

In [1]:
# Global Country Loop Experiments - Using Global Experiment Framework
import pandas as pd
import numpy as np
import os
import time
import sys
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

print("✅ Sklearn feature name warnings suppressed")

# Use 4 GPUs for this notebook
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0,1,2,3')
print("🔧 CUDA_VISIBLE_DEVICES =", os.environ.get('CUDA_VISIBLE_DEVICES'))

# Check GPU availability
import torch
print(f"\n🔧 PyTorch Version: {torch.__version__}")
print(f"🔧 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔧 GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"🔧 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"🔧 GPU Count: {torch.cuda.device_count()}")
else:
    print("⚠️ No GPU available, will use CPU (slower performance expected)")

# Import Global Experiment Runner
try:
    from model_trainer_global import run_global_experiments, GlobalExperimentRunner
    print("\n✅ Successfully imported Global Experiment Runner")
except ImportError:
    sys.path.append('../python modules')
    try:
        from model_trainer_global import run_global_experiments, GlobalExperimentRunner
        print("\n✅ Imported Global Experiment Runner via relative path")
    except ImportError:
        sys.path.append(r'd:\9_Projects\Transfer learning\python modules')
        from model_trainer_global import run_global_experiments, GlobalExperimentRunner
        print("\n✅ Imported Global Experiment Runner via absolute path")

# --- Configuration ---
data_dir = '../data'
processed_hdf5_path = os.path.join(data_dir, 'processed_feature_data.h5')
results_dir = '../results'
os.makedirs(results_dir, exist_ok=True)

# Verify file paths
print(f"\nCurrent working directory: {os.getcwd()}")
print(f"Data file path: {os.path.abspath(processed_hdf5_path)}")
print(f"File exists: {os.path.exists(processed_hdf5_path)}")

# Target column definition
TARGET_COLS = [
    'log_Scope1', 'log_Scope2', 'log_Scope3_upstream', 'log_Scope3_downLA', 
    'log_Scope3_prod', 'log_Scope_total'
]

# --- Load Global Data ---
print(f"\nLoading global feature-engineered data from {processed_hdf5_path}...")
try:
    df = pd.read_hdf(processed_hdf5_path, key='processed_features')
    
    # Global data quality check
    print("🔍 Global Data Quality Check:")
    
    # Ensure target columns exist
    for col in TARGET_COLS:
        if col not in df.columns:
            raise ValueError(f"Target column {col} not found in loaded data!")

    # Check geographic location column
    if 'loc' not in df.columns:
        geo_columns = [col for col in df.columns if col.lower() in ['country', 'location']]
        if geo_columns:
            df = df.rename(columns={geo_columns[0]: 'loc'})
            print(f"  - Renamed {geo_columns[0]} to 'loc'")
        else:
            raise ValueError("Missing country/region identifier column in data!")
    
    print(f"  - Found {df['loc'].nunique()} unique loc values in raw data")
    
    # Strict data cleaning
    initial_count = len(df)
    
    valid_mask = (
        df['loc'].notna() &
        (df['loc'] != '') &
        (df['loc'].astype(str).str.strip() != '') &
        (df['loc'].astype(str).str.len() >= 2) &
        (df['loc'].astype(str).str.isalpha()) &
        (df['loc'].astype(str).str.len() <= 10)
    )
    
    df = df[valid_mask].copy()
    removed_count = initial_count - len(df)
    
    print(f"  - Removed samples with anomalous loc values: {removed_count:,}")
    print(f"  - Found {df['loc'].nunique()} valid countries/regions after cleaning")

    # Get feature column names
    exclude_cols = TARGET_COLS + ['gvkey', 'fiscalyear', 'loc']
    other_exclude = [col for col in df.columns if 'sector' in col.lower() or 'gics' in col.lower()]
    exclude_cols.extend(other_exclude)
    
    FEATURE_COLS = [col for col in df.columns if col not in exclude_cols]
    
    print(f"  - Number of feature columns: {len(FEATURE_COLS)}")
    
    # Check target column NaN status
    for col in TARGET_COLS:
        nan_count = df[col].isnull().sum()
        valid_count = df[col].notnull().sum()
        print(f"  - {col}: {valid_count:,} valid, {nan_count:,} missing")
    
    # Remove rows where all targets are NaN
    initial_count_targets = len(df)
    df = df.dropna(subset=TARGET_COLS, how='all')
    removed_count_targets = initial_count_targets - len(df)
    
    print(f"  - Removed samples with all-NaN targets: {removed_count_targets:,}")
    print(f"  - Retained samples: {len(df):,}")
    
    print("\n✅ Global data loaded successfully!")
    print(f"Final data shape: {df.shape}")
    print(f"Final feature count: {len(FEATURE_COLS)}")
    print(f"Valid country count: {df['loc'].nunique()}")
    
    # Show top 15 countries by sample size
    country_counts = df['loc'].value_counts().head(15)
    print(f"\n📊 Top 15 Countries by Sample Size:")
    for i, (country, count) in enumerate(country_counts.items(), 1):
        print(f"  {i:2d}. '{country}': {count:,}")
    
except Exception as e:
    print(f"❌ Failed to load data: {e}")
    raise

✅ Sklearn feature name warnings suppressed
🔧 CUDA_VISIBLE_DEVICES = 0,1,2,3

🔧 PyTorch Version: 2.5.1+cu124
🔧 CUDA Available: True
🔧 GPU Device: NVIDIA A800-SXM4-80GB
🔧 GPU Memory: 79.3 GB
🔧 GPU Count: 4

✅ Imported Global Experiment Runner via relative path

Current working directory: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/notebooks
Data file path: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/data/processed_feature_data.h5
File exists: True

Loading global feature-engineered data from ../data/processed_feature_data.h5...
🔍 Global Data Quality Check:
  - Found 149 unique loc values in raw data
  - Removed samples with anomalous loc values: 24,271
  - Found 148 valid countries/regions after cleaning
  - Number of feature columns: 100
  - log_Scope1: 199,290 valid, 1,274,180 missing
  - log_Scope2: 199,290 valid, 1,274,180 missing
  - log_Scope3_upstream: 199,290 valid, 1,274,180 missing
  - log_Sco

In [2]:
# --- GPU-intensive Experiment Configuration ---

import time
import sys
import os

print("🔧 Adding python modules directory to path...")
sys.path.append(os.path.abspath('../python modules'))
print("✅ Path added")

try:
    from model_trainer_global import GlobalExperimentRunner, run_global_experiments
    print("✅ Successfully imported GlobalExperimentRunner and run_global_experiments")
except ImportError as e:
    print(f"❌ Import failed: {e}")

# 🎯 Experiment Configuration Parameters
EXPERIMENT_CONFIG = {
    # Experiment parameters
    'min_samples': 50,                      # Minimum sample size threshold
    'experiments': ['C', 'C_dev', 'D_lora', 'D_lora_dev', 'D_full', 'D_full_dev'],  # GPU-intensive experiments
    'random_state': 42,                     # Random seed
    
    # MLP Transfer Learning parameters (for Experiments C/C_dev)
    'n_finetune_trials': 25,                # MLP fine-tuning hyperparameter trials
    'pretrain_epochs': 50,                  # MLP pre-training epochs
    'finetune_epochs': 100,                 # MLP fine-tuning epochs
    
    # FT-Transformer parameters (for Experiments D_lora/D_full series)
    'pretrain_epochs_ftt': 60,              # FTT pre-training epochs
    'finetune_epochs_ftt': 40,              # FTT fine-tuning epochs
    'lora_rank': 8,                         # LoRA rank
    'lora_alpha': 16,                       # LoRA scaling coefficient
}

# Available experiment types
ALL_EXPERIMENTS = {
    'C': 'Transfer Learning - MLP global pre-training → single-country fine-tuning',
    'C_dev': 'Developed Countries Transfer Learning - MLP developed pre-training → fine-tuning',
    'D_lora': 'FT-Transformer LoRA - Global pre-training → LoRA fine-tuning',
    'D_lora_dev': 'FT-Transformer LoRA (Developed) - Developed pre-training → LoRA fine-tuning',
    'D_full': 'FT-Transformer Full - Global pre-training → full parameter fine-tuning',
    'D_full_dev': 'FT-Transformer Full (Developed) - Developed pre-training → full fine-tuning',
}

print("🌍 GPU-intensive Experiment Configuration:")
print("=" * 70)
print(f"Selected experiment types: {EXPERIMENT_CONFIG['experiments']}")
print(f"\n📋 Available Experiment Types (GPU-intensive):")
for exp_code, exp_desc in ALL_EXPERIMENTS.items():
    selected = "✓" if exp_code in EXPERIMENT_CONFIG['experiments'] else " "
    print(f"  [{selected}] {exp_code}: {exp_desc}")
print(f"\nMinimum sample threshold: {EXPERIMENT_CONFIG['min_samples']}")
print(f"MLP pre-training epochs: {EXPERIMENT_CONFIG['pretrain_epochs']}")
print(f"MLP fine-tuning epochs: {EXPERIMENT_CONFIG['finetune_epochs']}")
print(f"FTT pre-training epochs: {EXPERIMENT_CONFIG['pretrain_epochs_ftt']}")
print(f"FTT fine-tuning epochs: {EXPERIMENT_CONFIG['finetune_epochs_ftt']}")
print(f"LoRA rank: {EXPERIMENT_CONFIG['lora_rank']}, LoRA alpha: {EXPERIMENT_CONFIG['lora_alpha']}")
print("=" * 70)

# 🎯 Run Mode Selection
print(f"\n🎯 Available Run Modes:")
print("1. debug_mode    - Debug mode: Run on 3 major countries (USA, JPN, CHN)")
print("2. full_auto     - Full auto mode: Run on all countries meeting threshold")
print("3. custom_list   - Custom mode: Run on specified country list")

# 🔧 Select run mode here
RUN_MODE = "full_auto"  # Options: "debug_mode", "full_auto", "custom_list"

# Configure target countries and parameters based on run mode
if RUN_MODE == "debug_mode":
    print(f"\n🎯 Run Mode: {RUN_MODE} (Debug mode)")
    target_countries = ['USA', 'JPN', 'CHN']
    # Reduce epochs for debug mode
    EXPERIMENT_CONFIG['pretrain_epochs'] = 20
    EXPERIMENT_CONFIG['finetune_epochs'] = 30
    EXPERIMENT_CONFIG['pretrain_epochs_ftt'] = 30
    EXPERIMENT_CONFIG['finetune_epochs_ftt'] = 20
    EXPERIMENT_CONFIG['n_finetune_trials'] = 10
    print("🔧 Debug mode parameter adjustments:")
    print(f"  - MLP pre-training epochs: {EXPERIMENT_CONFIG['pretrain_epochs']} (reduced)")
    print(f"  - MLP fine-tuning epochs: {EXPERIMENT_CONFIG['finetune_epochs']} (reduced)")
    print(f"  - FTT pre-training epochs: {EXPERIMENT_CONFIG['pretrain_epochs_ftt']} (reduced)")
    print(f"  - FTT fine-tuning epochs: {EXPERIMENT_CONFIG['finetune_epochs_ftt']} (reduced)")
    
elif RUN_MODE == "full_auto":
    print(f"\n🎯 Run Mode: {RUN_MODE} (Full auto mode)")
    target_countries = "auto"
    print("🌍 Will automatically select all countries meeting sample threshold")
    print("📊 Using full parameters for experiments")
    
elif RUN_MODE == "custom_list":
    print(f"\n🎯 Run Mode: {RUN_MODE} (Custom mode)")
    target_countries = [
        'USA', 'JPN', 'CHN', 'GBR', 'KOR', 'TWN', 'IND', 'AUS'
    ]
    print(f"🎯 Custom country list: {target_countries}")
    print("📊 Using full parameters for experiments")
    
else:
    raise ValueError(f"Unsupported run mode: {RUN_MODE}")

print(f"Target countries: {target_countries}")

# --- Data Validation ---
print(f"\n{'='*60}")
print(f"📊 Pre-run Data Validation...")
print(f"{'='*60}")

print(f"📊 Global Data Statistics:")
print(f"  - Total samples: {len(df):,}")
print(f"  - Total countries: {df['loc'].nunique()}")
print(f"  - Feature count: {len(FEATURE_COLS)}")
print(f"  - Target count: {len(TARGET_COLS)}")

# Data validation based on run mode
if RUN_MODE == "full_auto":
    print(f"\n🌍 Full auto mode: Filtering eligible countries...")
    print(f"Filter criteria: sample_count >= {EXPERIMENT_CONFIG['min_samples']}")
    
    valid_countries = []
    country_stats = []
    
    for country in df['loc'].unique():
        country_data = df[df['loc'] == country]
        country_samples = len(country_data)
        
        all_targets_valid = country_data[TARGET_COLS].notnull().all(axis=1)
        feature_complete = country_data[FEATURE_COLS].notnull().all(axis=1)
        usable_mask = all_targets_valid & feature_complete
        usable_samples = usable_mask.sum()
        
        country_stats.append({
            'country': country,
            'total_samples': country_samples,
            'usable_samples': usable_samples,
            'usable_rate': usable_samples / country_samples * 100 if country_samples > 0 else 0
        })
        
        if usable_samples >= EXPERIMENT_CONFIG['min_samples']:
            valid_countries.append(country)
    
    country_stats.sort(key=lambda x: x['usable_samples'], reverse=True)
    
    print(f"\n📊 Country Data Quality Statistics (sorted by usable samples):")
    print(f"{'Country':<8} {'Total':<12} {'Usable':<12} {'Rate':<10} {'Status':<10}")
    print("-" * 55)
    
    for i, stats in enumerate(country_stats[:20]):
        country = stats['country']
        total = stats['total_samples']
        usable = stats['usable_samples']
        rate = stats['usable_rate']
        status = "✅ Pass" if usable >= EXPERIMENT_CONFIG['min_samples'] else "❌ Fail"
        print(f"{country:<8} {total:<12,} {usable:<12,} {rate:<9.1f}% {status}")
    
    if len(country_stats) > 20:
        print(f"... {len(country_stats) - 20} more countries not shown")
    
    target_countries = valid_countries
    print(f"\n✅ Auto-selected {len(target_countries)} eligible countries")
    
else:
    print(f"\n📋 Target Country Data Validation:")
    valid_target_countries = []
    
    for country in target_countries:
        if country in df['loc'].values:
            country_data = df[df['loc'] == country]
            country_samples = len(country_data)
            
            all_targets_valid = country_data[TARGET_COLS].notnull().all(axis=1)
            feature_complete = country_data[FEATURE_COLS].notnull().all(axis=1)
            usable_mask = all_targets_valid & feature_complete
            usable_samples = usable_mask.sum()
            
            print(f"  - {country}: {country_samples:,} total, {usable_samples:,} usable ({usable_samples/country_samples*100:.1f}%)")
            
            if usable_samples >= EXPERIMENT_CONFIG['min_samples']:
                valid_target_countries.append(country)
                print(f"    ✅ Meets minimum sample requirement")
            else:
                print(f"    ❌ Does not meet minimum sample requirement (need {EXPERIMENT_CONFIG['min_samples']})")
        else:
            print(f"  - {country}: ❌ Not in data")
    
    target_countries = valid_target_countries

print(f"\n✅ Final valid target country count: {len(target_countries)}")
if len(target_countries) <= 10:
    print(f"✅ Final valid target countries: {target_countries}")
else:
    print(f"✅ First 10 countries: {target_countries[:10]}")

# Estimate experiment time
total_experiments = len(target_countries) * len(EXPERIMENT_CONFIG['experiments'])
estimated_time_per_exp = 5 if RUN_MODE == "debug_mode" else 15  # GPU experiments take longer
estimated_total_time = total_experiments * estimated_time_per_exp

print(f"\n⏱️ Experiment Scale Estimate:")
print(f"  - Target countries: {len(target_countries)}")
print(f"  - Experiment types: {len(EXPERIMENT_CONFIG['experiments'])}")
print(f"  - Total experiments: {total_experiments}")
print(f"  - Estimated total time: {estimated_total_time:.0f} minutes ({estimated_total_time/60:.1f} hours)")

print(f"\n✅ Configuration and validation phase complete!")

🔧 Adding python modules directory to path...
✅ Path added
✅ Successfully imported GlobalExperimentRunner and run_global_experiments
🌍 GPU-intensive Experiment Configuration:
Selected experiment types: ['C', 'C_dev', 'D_lora', 'D_lora_dev', 'D_full', 'D_full_dev']

📋 Available Experiment Types (GPU-intensive):
  [✓] C: Transfer Learning - MLP global pre-training → single-country fine-tuning
  [✓] C_dev: Developed Countries Transfer Learning - MLP developed pre-training → fine-tuning
  [✓] D_lora: FT-Transformer LoRA - Global pre-training → LoRA fine-tuning
  [✓] D_lora_dev: FT-Transformer LoRA (Developed) - Developed pre-training → LoRA fine-tuning
  [✓] D_full: FT-Transformer Full - Global pre-training → full parameter fine-tuning
  [✓] D_full_dev: FT-Transformer Full (Developed) - Developed pre-training → full fine-tuning

Minimum sample threshold: 50
MLP pre-training epochs: 50
MLP fine-tuning epochs: 100
FTT pre-training epochs: 60
FTT fine-tuning epochs: 40
LoRA rank: 8, LoRA alpha


📊 Country Data Quality Statistics (sorted by usable samples):
Country  Total        Usable       Rate       Status    
-------------------------------------------------------
USA      38,379       15,514       40.4     % ✅ Pass
CHN      23,057       13,988       60.7     % ✅ Pass
JPN      23,682       11,099       46.9     % ✅ Pass
KOR      10,737       5,476        51.0     % ✅ Pass
TWN      8,959        4,315        48.2     % ✅ Pass
IND      7,739        3,618        46.8     % ✅ Pass
HKG      6,089        2,678        44.0     % ✅ Pass
GBR      10,881       2,326        21.4     % ✅ Pass
AUS      6,644        2,220        33.4     % ✅ Pass
CAN      5,369        2,115        39.4     % ✅ Pass
FRA      4,011        1,397        34.8     % ✅ Pass
SWE      3,163        1,355        42.8     % ✅ Pass
DEU      3,676        1,349        36.7     % ✅ Pass
MYS      2,691        1,177        43.7     % ✅ Pass
THA      2,453        1,155        47.1     % ✅ Pass
CHE      3,051        1,031  

In [ ]:
# --- Experiment Execution Cell ---

# Configure Optuna logging
import optuna
import logging
import pickle
from datetime import datetime
import multiprocessing as mp
import torch
import os
import sys

from gpu_cd_worker import gpu_cd_worker

# Checkpoint helpers (for resume)
try:
    from checkpoint_utils import list_checkpoint_runs, pick_checkpoint_run, load_checkpoint_results
except ImportError:
    sys.path.append(os.path.abspath('../python modules'))
    from checkpoint_utils import list_checkpoint_runs, pick_checkpoint_run, load_checkpoint_results

# Use checkpoint-enabled runner for serial fallback
try:
    from model_trainer_all import run_global_experiments as run_global_experiments_with_ckpt
except ImportError:
    sys.path.append(os.path.abspath('../python modules'))
    from model_trainer_all import run_global_experiments as run_global_experiments_with_ckpt

# Clean and reconfigure Optuna logging
optuna_logger = optuna.logging.get_logger("optuna")
for handler in optuna_logger.handlers[:]:
    optuna_logger.removeHandler(handler)

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("🔧 Log configuration optimized")

# --- Checkpoint configuration ---
CHECKPOINT_BASE_DIR = os.path.join(results_dir, "gpu_cd_checkpoints")
RESUME_FROM_CHECKPOINTS = True
CHECKPOINT_RUN_ID = None  # Set to existing run_id string to resume, or leave None to auto-pick latest
PREFER_LATEST_COMMON_RUN = True

def _find_latest_common_run(base_dir, worker_ids):
    if not worker_ids:
        return None
    common = None
    for wid in worker_ids:
        worker_dir = os.path.join(base_dir, f"worker_{wid}")
        runs = [r.signature for r in list_checkpoint_runs(worker_dir)]
        if common is None:
            common = set(runs)
        else:
            common &= set(runs)
    if not common:
        return None
    runs_sorted = list_checkpoint_runs(os.path.join(base_dir, f"worker_{worker_ids[0]}"))
    for run in runs_sorted:
        if run.signature in common:
            return run.signature
    return None

def _latest_serial_run(base_dir):
    runs = list_checkpoint_runs(base_dir)
    return runs[0].signature if runs else None

# --- Helper: build results dataframe (GPU experiments) ---
def build_gpu_results_df(experiment_results, target_cols):
    results_data = []
    for result in experiment_results:
        country = result.get('country', 'Unknown')
        exp_type_raw = result.get('experiment_type', 'Unknown')
        
        exp_type = exp_type_raw
        if 'Transfer_Learning' in exp_type_raw and 'FTT' not in exp_type_raw:
            exp_type = 'C_dev' if 'Developed' in exp_type_raw else 'C'
        elif 'FTT_LoRA' in exp_type_raw:
            exp_type = 'D_lora_dev' if 'Developed' in exp_type_raw else 'D_lora'
        elif 'FTT_Full' in exp_type_raw:
            exp_type = 'D_full_dev' if 'Developed' in exp_type_raw else 'D_full'
        
        r2_scores = {}
        r2_avg = result.get('R2_Average', np.nan)
        for target in target_cols:
            r2_key = f'R2_{target}'
            r2_scores[target] = result.get(r2_key, np.nan)
        
        mae_values = [result.get(f'MAE_{target}', np.nan) for target in target_cols]
        rmse_values = [result.get(f'RMSE_{target}', np.nan) for target in target_cols]
        mae_avg = np.nanmean(mae_values) if mae_values else np.nan
        rmse_avg = np.nanmean(rmse_values) if rmse_values else np.nan
        
        training_samples = result.get('training_samples', 0)
        finetune_samples = result.get('finetune_samples', 0)
        
        results_data.append({
            'country': country,
            'experiment_type': exp_type,
            'experiment_type_full': exp_type_raw,
            'R2_Average': r2_avg,
            'MAE_Average': mae_avg,
            'RMSE_Average': rmse_avg,
            'training_samples': training_samples,
            'finetune_samples': finetune_samples,
            **r2_scores
        })
    
    return pd.DataFrame(results_data)

# --- Helper: save results to Excel (English format) ---
def save_results_excel(results_df, results_dir, prefix, metadata):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_path = os.path.join(results_dir, f"{prefix}_{timestamp}.xlsx")
    
    summary_by_exp = results_df.groupby('experiment_type').agg({
        'R2_Average': ['count', 'mean', 'std', 'min', 'max'],
        'MAE_Average': 'mean',
        'RMSE_Average': 'mean'
    }).round(4)
    
    summary_by_country = results_df.groupby('country').agg({
        'R2_Average': ['count', 'mean', 'std']
    }).round(4)
    
    metadata_df = pd.DataFrame(list(metadata.items()), columns=['Field', 'Value'])
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='Results', index=False)
        summary_by_exp.to_excel(writer, sheet_name='Summary_By_Experiment')
        summary_by_country.to_excel(writer, sheet_name='Summary_By_Country')
        metadata_df.to_excel(writer, sheet_name='Metadata', index=False)
    
    return excel_path

# --- Multi-GPU execution helpers ---
def _split_list_round_robin(items, n):
    return [items[i::n] for i in range(n)]


# --- Run experiments (multi-GPU parallel by country) ---
RUN_GPU_EXPERIMENTS = True  # set True to run
USE_MULTI_GPU = True
GPU_IDS = [0, 1, 2, 3]
NUM_WORKERS = len(GPU_IDS)

if RUN_GPU_EXPERIMENTS:
    if isinstance(target_countries, str) and target_countries == "auto":
        raise ValueError("target_countries should be resolved before execution.")

    if not target_countries:
        print("⚠️ No valid target countries. Skipping run.")
        experiment_results = []
        experiment_duration = 0
    else:
        # Resolve run_id for checkpoint resume
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        if RESUME_FROM_CHECKPOINTS:
            resolved_run_id = CHECKPOINT_RUN_ID
            if resolved_run_id is None and PREFER_LATEST_COMMON_RUN and USE_MULTI_GPU:
                resolved_run_id = _find_latest_common_run(CHECKPOINT_BASE_DIR, GPU_IDS)
            if resolved_run_id is None and PREFER_LATEST_COMMON_RUN and not USE_MULTI_GPU:
                resolved_run_id = _latest_serial_run(os.path.join(CHECKPOINT_BASE_DIR, "serial"))
            if resolved_run_id:
                run_id = resolved_run_id
                print(f"🔁 Resume enabled. Using checkpoint run_id={run_id}")
            else:
                print("⚠️ Resume enabled but no checkpoints found; starting a new run")

        df_cache_path = os.path.join(results_dir, f"gpu_cd_df_cache_{run_id}.pkl")
        print(f"💾 Caching cleaned dataframe to: {df_cache_path}")
        df.to_pickle(df_cache_path)

        if USE_MULTI_GPU and torch.cuda.is_available() and torch.cuda.device_count() >= NUM_WORKERS:
            ctx = mp.get_context("spawn")
            print(f"🚀 Multi-GPU mode enabled: {NUM_WORKERS} workers on GPUs {GPU_IDS} (start_method=spawn)")
            chunks = _split_list_round_robin(target_countries, NUM_WORKERS)
            for i, chunk in enumerate(chunks):
                print(f"  - Worker {i} -> {len(chunk)} countries")

            result_q = ctx.Queue()
            procs = []

            start_time = time.time()
            for worker_id, (gpu_id, countries_chunk) in enumerate(zip(GPU_IDS, chunks)):
                p = ctx.Process(
                    target=gpu_cd_worker,
                    args=(
                        result_q,
                        worker_id,
                        gpu_id,
                        countries_chunk,
                        EXPERIMENT_CONFIG,
                        df_cache_path,
                        FEATURE_COLS,
                        TARGET_COLS,
                        results_dir,
                        run_id
                    )
                )
                p.start()
                procs.append(p)

            all_results = []
            errors = []
            for _ in procs:
                msg = result_q.get()
                if msg.get('error'):
                    errors.append((msg.get('worker_id'), msg.get('error')))
                all_results.extend(msg.get('results', []))

            for p in procs:
                p.join()

            end_time = time.time()
            experiment_results = all_results
            experiment_duration = (end_time - start_time) / 60

            if errors:
                print("⚠️ Some workers failed:")
                for wid, err in errors:
                    print(f"  - worker {wid}: {err}")
        else:
            print("⚠️ Multi-GPU unavailable. Falling back to serial execution.")
            start_time = time.time()
            serial_ckpt_dir = os.path.join(CHECKPOINT_BASE_DIR, "serial")
            experiment_results = run_global_experiments_with_ckpt(
                df=df,
                features=FEATURE_COLS,
                targets=TARGET_COLS,
                countries=target_countries,
                min_samples=EXPERIMENT_CONFIG.get('min_samples', 50),
                experiments=EXPERIMENT_CONFIG.get('experiments', []),
                n_finetune_trials=EXPERIMENT_CONFIG.get('n_finetune_trials', 25),
                pretrain_epochs=EXPERIMENT_CONFIG.get('pretrain_epochs', 50),
                finetune_epochs=EXPERIMENT_CONFIG.get('finetune_epochs', 100),
                ft_pretrain_epochs=EXPERIMENT_CONFIG.get('pretrain_epochs_ftt', 60),
                ft_finetune_epochs=EXPERIMENT_CONFIG.get('finetune_epochs_ftt', 40),
                lora_rank=EXPERIMENT_CONFIG.get('lora_rank', 8),
                lora_alpha=EXPERIMENT_CONFIG.get('lora_alpha', 16),
                checkpoint_dir=serial_ckpt_dir,
                checkpoint_run_id=run_id,
                resume=RESUME_FROM_CHECKPOINTS
            )
            end_time = time.time()
            experiment_duration = (end_time - start_time) / 60

        print(f"\n🎉 GPU experiments complete!")
        print(f"Total time: {experiment_duration:.2f} minutes")
        print(f"Completed experiments: {len(experiment_results)}")

        # Immediate Excel export
        if experiment_results:
            results_df = build_gpu_results_df(experiment_results, TARGET_COLS)
            metadata = {
                "Run_Mode": RUN_MODE,
                "Experiments": ", ".join(EXPERIMENT_CONFIG['experiments']),
                "Min_Samples": EXPERIMENT_CONFIG['min_samples'],
                "MLP_Pretrain_Epochs": EXPERIMENT_CONFIG['pretrain_epochs'],
                "MLP_Finetune_Epochs": EXPERIMENT_CONFIG['finetune_epochs'],
                "FTT_Pretrain_Epochs": EXPERIMENT_CONFIG['pretrain_epochs_ftt'],
                "FTT_Finetune_Epochs": EXPERIMENT_CONFIG['finetune_epochs_ftt'],
                "LoRA_Rank": EXPERIMENT_CONFIG['lora_rank'],
                "LoRA_Alpha": EXPERIMENT_CONFIG['lora_alpha'],
                "Target_Countries": len(target_countries),
                "Start_Time": datetime.fromtimestamp(start_time).strftime("%Y-%m-%d %H:%M:%S"),
                "End_Time": datetime.fromtimestamp(end_time).strftime("%Y-%m-%d %H:%M:%S"),
                "Total_Time_Min": round(experiment_duration, 2),
                "Checkpoint_Run_ID": run_id
            }
            excel_path = save_results_excel(
                results_df,
                results_dir,
                prefix="Global_Carbon_Emissions_Experiment_Results_GPU_CD",
                metadata=metadata
            )
            print(f"💾 Results saved immediately to: {excel_path}")

        # Summary
        successful = [r for r in experiment_results if r.get('status') != 'failed' and not np.isnan(r.get('R2_Average', np.nan))]
        failed = [r for r in experiment_results if r.get('status') == 'failed' or np.isnan(r.get('R2_Average', np.nan))]
        print(f"\n📊 Experiment Results Overview:")
        print(f"  - Successful experiments: {len(successful)}")
        print(f"  - Failed experiments: {len(failed)}")

        exp_types = {}
        for result in experiment_results:
            exp_type = result.get('experiment_type', 'Unknown')
            if exp_type not in exp_types:
                exp_types[exp_type] = {'success': 0, 'failed': 0}

            if result.get('status') != 'failed' and not np.isnan(result.get('R2_Average', np.nan)):
                exp_types[exp_type]['success'] += 1
            else:
                exp_types[exp_type]['failed'] += 1

        print(f"\n📈 Statistics by Experiment Type:")
        for exp_type, counts in exp_types.items():
            total = counts['success'] + counts['failed']
            success_rate = counts['success'] / total * 100 if total > 0 else 0
            print(f"  - {exp_type}: {counts['success']}/{total} success ({success_rate:.1f}%)")

else:
    print("⚠️ RUN_GPU_EXPERIMENTS is False. Skipping run.")
    experiment_results = []
    experiment_duration = 0

print(f"\n✅ Experiment execution phase complete!")
if experiment_results:
    print(f"✅ Experiment data ready for analysis")
    print(f"📊 Variable 'experiment_results' contains {len(experiment_results)} experiment results")
    print(f"⏱️ Variable 'experiment_duration' records total time: {experiment_duration:.2f} minutes")
else:
    print(f"⚠️ No experiment results, please check data and configuration")

🔧 Log configuration optimized
💾 Caching cleaned dataframe to: ../results/gpu_cd_df_cache_20260206_090932.pkl
🚀 Multi-GPU mode enabled: 4 workers on GPUs [0, 1, 2, 3] (start_method=spawn)
  - Worker 0 -> 14 countries
  - Worker 1 -> 14 countries
  - Worker 2 -> 14 countries
  - Worker 3 -> 13 countries
🌍 GlobalExperimentRunner initialized. Compute device: GPU
📊 Found countries/regions
🔍 Analyzing global data quality...
🌍 GlobalExperimentRunner initialized. Compute device: GPU
📊 Found countries/regions
🔍 Analyzing global data quality...
🌍 GlobalExperimentRunner initialized. Compute device: GPU
🌍 GlobalExperimentRunner initialized. Compute device: GPU
📊 Found countries/regions
🔍 Analyzing global data quality...
📊 Found countries/regions
🔍 Analyzing global data quality...
📈 Top 10 countries by sample size:
   1. USA: 15,514valid samples (all labels) (40.4%)
   2. CHN: 13,988valid samples (all labels) (60.7%)
   3. JPN: 11,099valid samples (all labels) (46.9%)
   4. KOR: 5,476valid samples 

globalexperimentprogress:   0%|          | 0/13 [00:00<?, ?it/s]

    📊 pre-trainingsamples: 69,709, fine-tuning samples: 15,514
    📊 pre-trainingsamples: 84,779, fine-tuning samples: 444
    📊 pre-trainingsamples: 83,108, fine-tuning samples: 2,115
    📊 pre-trainingsamples: 84,439, fine-tuning samples: 784
    🏗️ pre-trainingcomplete: loss = 4.0844
    📊 experimentC(Transfer Learning) - USA Dataset split and sources:
      📍 Data source strategy: global datapre-training → target countryfine-tuning (MLP pre-training + fine-tuning)
      🌍 Phase 1 — source-domain pre-training:
        🏗️ Pre-training set: 69,709 samples - Source: global data (excluding USA)
      🎯 Phase 2 — USA fine-tuning and testing:
        📈 Total usable samples: 15,514 - Source: USA country data
        🏋️ fine-tuningTraining set: 9,222 samples (59.4%) - Source: USA country data
        ✅ fine-tuningvalidation set: 3,294 samples (21.2%) - Source: USA country data
        🎯 test set: 2,998 samples (19.3%) - Source: USA country data
      💡 Description: pre-train on global data,

[I 2026-02-06 09:16:48,574] A new study created in memory with name: no-name-51e120ee-166a-442e-a044-78592e01fb29

  0%|          | 0/25 [00:00<?, ?it/s]

    🏗️ pre-trainingcomplete: loss = 3.9707
    📊 experimentC(Transfer Learning) - ISR Dataset split and sources:
      📍 Data source strategy: global datapre-training → target countryfine-tuning (MLP pre-training + fine-tuning)
      🌍 Phase 1 — source-domain pre-training:
        🏗️ Pre-training set: 84,439 samples - Source: global data (excluding ISR)
      🎯 Phase 2 — ISR fine-tuning and testing:
        📈 Total usable samples: 784 - Source: ISR country data
        🏋️ fine-tuningTraining set: 440 samples (56.1%) - Source: ISR country data
        ✅ fine-tuningvalidation set: 176 samples (22.4%) - Source: ISR country data
        🎯 test set: 168 samples (21.4%) - Source: ISR country data
      💡 Description: pre-train on global data, then fine-tune on target-country data to adapt to local features.


...

                                                                
Best trial: 1. Best value: 0.783815:  12%|█▏        | 3/25 [06:00<44:08, 120.39s/it]

[I 2026-02-06 09:22:49,558] Trial 2 finished with value: 0.740349551041921 and parameters: {'lr': 2.0513382630874486e-05, 'weight_decay': 2.9375384576328313e-06}. Best is trial 1 with value: 0.7838146090507507.


globalexperimentprogress:   0%|          | 0/14 [13:33<?, ?it/s]
Process SpawnProcess-3:
globalexperimentprogress:   0%|          | 0/13 [13:33<?, ?it/s]
Process SpawnProcess-4:
globalexperimentprogress:   0%|          | 0/14 [13:33<?, ?it/s]
Process SpawnProcess-2:
Traceback (most recent call last):
  File "/hpc2hdd/home/yjiang909/.conda/envs/transfer_learning_env/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/hpc2hdd/home/yjiang909/.conda/envs/transfer_learning_env/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/notebooks/../python modules/gpu_cd_worker.py", line 201, in gpu_cd_worker
    results = run_global_experiments(
  File "/hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/notebooks/../python modules/model_trainer_all.py", line 2766, in run_globa

[W 2026-02-06 09:23:10,970] Trial 3 failed with parameters: {'lr': 1.3066739238053272e-05, 'weight_decay': 0.0003967605077052988} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/hpc2hdd/home/yjiang909/.conda/envs/transfer_learning_env/lib/python3.10/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "/hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/notebooks/../python modules/model_trainer_all.py", line 1592, in objective
    train_epoch(model, train_loader, optimizer, criterion, self.device)
  File "/hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/notebooks/../python modules/model_trainer_all.py", line 376, in train_epoch
    optimizer.step()                       # Update params: adjust weights based on gradients
  File "/hpc2hdd/home/yjiang909/.conda/envs/transfer_learning_env/lib/python3.10/site-packages

KeyboardInterrupt: 

In [ ]:
# --- Experiment Results Visualization ---

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Set font and plot style
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('default')
sns.set_palette("husl")

print("🎨 Generating experiment results visualization...")
print("=" * 60)

if 'experiment_results' in locals() and len(experiment_results) > 0:
    print(f"✅ Found {len(experiment_results)} experiment results")
    
    # --- Process and organize experiment result data ---
    print("\n📊 Processing experiment result data...")
    
    results_data = []
    for result in experiment_results:
        country = result.get('country', 'Unknown')
        exp_type_raw = result.get('experiment_type', 'Unknown')
        
        # Standardize experiment type names
        exp_type = exp_type_raw
        if 'Transfer_Learning' in exp_type_raw and 'FTT' not in exp_type_raw:
            if 'Developed' in exp_type_raw:
                exp_type = 'C_dev'
            else:
                exp_type = 'C'
        elif 'FTT_LoRA' in exp_type_raw:
            if 'Developed' in exp_type_raw:
                exp_type = 'D_lora_dev'
            else:
                exp_type = 'D_lora'
        elif 'FTT_Full' in exp_type_raw:
            if 'Developed' in exp_type_raw:
                exp_type = 'D_full_dev'
            else:
                exp_type = 'D_full'
        
        # Extract R² scores
        r2_scores = {}
        r2_avg = result.get('R2_Average', np.nan)
        
        for target in TARGET_COLS:
            r2_key = f'R2_{target}'
            r2_scores[target] = result.get(r2_key, np.nan)
        
        # Extract other metrics
        mae_values = [result.get(f'MAE_{target}', np.nan) for target in TARGET_COLS]
        rmse_values = [result.get(f'RMSE_{target}', np.nan) for target in TARGET_COLS]
        
        mae_avg = np.nanmean(mae_values) if mae_values else np.nan
        rmse_avg = np.nanmean(rmse_values) if rmse_values else np.nan
        
        training_samples = result.get('training_samples', 0)
        finetune_samples = result.get('finetune_samples', 0)
        
        results_data.append({
            'country': country,
            'experiment_type': exp_type,
            'experiment_type_full': exp_type_raw,
            'R2_Average': r2_avg,
            'MAE_Average': mae_avg,
            'RMSE_Average': rmse_avg,
            'training_samples': training_samples,
            'finetune_samples': finetune_samples,
            **r2_scores
        })
    
    results_df = pd.DataFrame(results_data)
    print(f"✅ Successfully processed {len(results_df)} experiment results")
    
    # Display results overview
    print(f"\n📈 Results Overview:")
    print(f"  - Countries involved: {results_df['country'].nunique()}")
    print(f"  - Experiment types: {list(results_df['experiment_type'].unique())}")
    valid_r2 = results_df['R2_Average'].dropna()
    if len(valid_r2) > 0:
        print(f"  - R² range: {valid_r2.min():.3f} - {valid_r2.max():.3f}")
    
    # --- Start Plotting ---
    print(f"\n🎨 Creating visualization charts...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: R² Heatmap by Country and Experiment Type
    ax1 = axes[0, 0]
    heatmap_data = results_df.pivot(index='country', columns='experiment_type', values='R2_Average')
    sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
               center=0.5, vmin=0, vmax=1, ax=ax1, cbar_kws={'label': 'R² Score'})
    ax1.set_title('R² Comparison Heatmap (GPU Experiments)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Experiment Type')
    ax1.set_ylabel('Country')
    
    # Plot 2: Box Plot by Experiment Type
    ax2 = axes[0, 1]
    exp_types_sorted = sorted(results_df['experiment_type'].unique())
    box_data = [results_df[results_df['experiment_type'] == exp]['R2_Average'].dropna() 
               for exp in exp_types_sorted if len(results_df[results_df['experiment_type'] == exp]['R2_Average'].dropna()) > 0]
    labels = [exp for exp in exp_types_sorted if len(results_df[results_df['experiment_type'] == exp]['R2_Average'].dropna()) > 0]
    if box_data:
        bp = ax2.boxplot(box_data, labels=labels, patch_artist=True)
        colors = plt.cm.Set3(np.linspace(0, 1, len(bp['boxes'])))
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
    ax2.set_title('Performance Distribution by Experiment Type', fontsize=14, fontweight='bold')
    ax2.set_ylabel('R² Score')
    ax2.set_xlabel('Experiment Type')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: MLP vs FT-Transformer Comparison
    ax3 = axes[1, 0]
    mlp_results = results_df[results_df['experiment_type'].isin(['C', 'C_dev'])]
    ftt_results = results_df[results_df['experiment_type'].isin(['D_lora', 'D_lora_dev', 'D_full', 'D_full_dev'])]
    
    if len(mlp_results) > 0 and len(ftt_results) > 0:
        x = np.arange(results_df['country'].nunique())
        width = 0.35
        countries = sorted(results_df['country'].unique())
        
        mlp_means = [mlp_results[mlp_results['country'] == c]['R2_Average'].mean() for c in countries]
        ftt_means = [ftt_results[ftt_results['country'] == c]['R2_Average'].mean() for c in countries]
        
        ax3.bar(x - width/2, mlp_means, width, label='MLP (C/C_dev)', color='steelblue')
        ax3.bar(x + width/2, ftt_means, width, label='FT-Transformer (D series)', color='coral')
        ax3.set_xticks(x)
        ax3.set_xticklabels(countries)
        ax3.legend()
    ax3.set_title('MLP vs FT-Transformer Performance', fontsize=14, fontweight='bold')
    ax3.set_ylabel('Average R² Score')
    ax3.set_xlabel('Country')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Plot 4: LoRA vs Full Fine-tuning Comparison
    ax4 = axes[1, 1]
    lora_results = results_df[results_df['experiment_type'].isin(['D_lora', 'D_lora_dev'])]
    full_results = results_df[results_df['experiment_type'].isin(['D_full', 'D_full_dev'])]
    
    if len(lora_results) > 0 and len(full_results) > 0:
        countries = sorted(results_df['country'].unique())
        x = np.arange(len(countries))
        width = 0.35
        
        lora_means = [lora_results[lora_results['country'] == c]['R2_Average'].mean() for c in countries]
        full_means = [full_results[full_results['country'] == c]['R2_Average'].mean() for c in countries]
        
        ax4.bar(x - width/2, lora_means, width, label='LoRA Fine-tuning', color='mediumseagreen')
        ax4.bar(x + width/2, full_means, width, label='Full Fine-tuning', color='mediumpurple')
        ax4.set_xticks(x)
        ax4.set_xticklabels(countries)
        ax4.legend()
    ax4.set_title('LoRA vs Full Fine-tuning Performance', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Average R² Score')
    ax4.set_xlabel('Country')
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'gpu_cd_experiments_results.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # --- Generate Numerical Summary ---
    print(f"\n📊 Detailed Numerical Summary:")
    
    print(f"\n📈 Summary by Experiment Type:")
    summary_by_exp = results_df.groupby('experiment_type').agg({
        'R2_Average': ['count', 'mean', 'std', 'min', 'max'],
        'MAE_Average': 'mean',
        'RMSE_Average': 'mean'
    }).round(4)
    print(summary_by_exp)
    
    print(f"\n🌍 Summary by Country:")
    summary_by_country = results_df.groupby('country').agg({
        'R2_Average': ['count', 'mean', 'std']
    }).round(4)
    print(summary_by_country)
    
    # Transfer learning effect analysis
    print(f"\n🔄 Transfer Learning Effect Analysis:")
    for country in results_df['country'].unique():
        country_data = results_df[results_df['country'] == country]
        if len(country_data) > 1:
            best_exp = country_data.loc[country_data['R2_Average'].idxmax()]
            print(f"  {country}: Best = {best_exp['experiment_type']} (R² = {best_exp['R2_Average']:.4f})")
    
    print(f"\n✅ Visualization analysis complete!")
    
else:
    print("❌ No experiment results data found")

print(f"\n🎉 Visualization module complete!")

In [ ]:
# --- Save Results to CSV ---

print("💾 Saving experiment results...")
print("=" * 60)

if 'results_df' in locals() and len(results_df) > 0:
    # Save full results
    output_path = os.path.join(results_dir, 'gpu_cd_experiments_full_results.csv')
    results_df.to_csv(output_path, index=False)
    print(f"✅ Full results saved to: {output_path}")
    
    # Save summary by experiment type
    summary_path = os.path.join(results_dir, 'gpu_cd_experiments_summary.csv')
    summary_by_exp = results_df.groupby('experiment_type').agg({
        'R2_Average': ['count', 'mean', 'std', 'min', 'max'],
        'MAE_Average': 'mean',
        'RMSE_Average': 'mean'
    }).round(4)
    summary_by_exp.to_csv(summary_path)
    print(f"✅ Summary saved to: {summary_path}")
    
    print(f"\n📊 Saved Results:")
    print(f"  - Total experiments: {len(results_df)}")
    print(f"  - Countries: {results_df['country'].nunique()}")
    print(f"  - Experiment types: {list(results_df['experiment_type'].unique())}")
    
    # Additional analysis: Best method per country
    print(f"\n🏆 Best Performing Method per Country:")
    for country in sorted(results_df['country'].unique()):
        country_data = results_df[results_df['country'] == country]
        if len(country_data) > 0:
            best_idx = country_data['R2_Average'].idxmax()
            best = country_data.loc[best_idx]
            print(f"  {country}: {best['experiment_type']} (R² = {best['R2_Average']:.4f})")
else:
    print("❌ No results to save")

print(f"\n🎉 03b GPU C/D Experiments Complete!")